In [1]:
import os
print(os.path.exists("models/dangerous.onnx"))  # 应输出 True


True


In [2]:
import tensorrt as trt

TRT_LOGGER = trt.Logger(trt.Logger.INFO)
builder = trt.Builder(TRT_LOGGER)
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
parser = trt.OnnxParser(network, TRT_LOGGER)

with open("models/dangerous.onnx", "rb") as f:
    if not parser.parse(f.read()):
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise RuntimeError("ONNX解析失败")

config = builder.create_builder_config()
config.max_workspace_size = 8 << 30
if builder.platform_has_fast_fp16:
    config.set_flag(trt.BuilderFlag.FP16)

serialized_engine = builder.build_serialized_network(network, config)
with open("models/dangerous.engine", "wb") as f:
    f.write(serialized_engine)


AttributeError: type object 'tensorrt_bindings.tensorrt.NetworkDefinitionCreati' has no attribute 'EXPLICIT_BATCH'

In [3]:
with open("models/dangerous.engine", "rb") as f:
    engine_data = f.read()
with trt.Runtime(trt.Logger(trt.Logger.WARNING)) as runtime:
    engine = runtime.deserialize_cuda_engine(engine_data)
    context = engine.create_execution_context()



In [4]:
import pycuda.driver as cuda
import numpy as np
import cv2
from datetime import datetime
import os, sys, logging, json
from config.detection_config import MODEL_CONFIG, SAVE_CONFIG, LOG_CONFIG
from myutils.logger import setup_logger

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))
logger = setup_logger('detection', LOG_CONFIG['base_dir'], LOG_CONFIG['sub_dirs']['detection'])

cuda_initialized = False
device = None
context = None

def initialize_cuda():
    global cuda_initialized, device, context
    if not cuda_initialized:
        cuda.init()
        device = cuda.Device(0)
        context = device.make_context()
        cuda_initialized = True

def release_cuda():
    global cuda_initialized, context, device
    if cuda_initialized and context:
        context.detach()
        context = None
        device = None
        cuda_initialized = False

initialize_cuda()
print('CUDA initialized:', cuda_initialized)


ModuleNotFoundError: No module named 'pycuda'